# Notebook Competitivo — Clasificación Temporal de Texto

Clasificador de textos históricos por década usando DeBERTa-v3 con validación cruzada, ensemble y features temporales.

## 1. Configuración y librerías

In [1]:
import os
import re
import gc
import json
import math
import string
import warnings
from copy import deepcopy
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding
)

warnings.filterwarnings('ignore')

# ── Reproducibilidad ─────────────────────────────────────────────────────
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ── Configuración ────────────────────────────────────────────────────────
class CFG:
    model_name = "microsoft/deberta-v3-base"
    max_len = 256
    batch_size = 16
    epochs = 4
    lr = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    label_smoothing = 0.1
    n_folds = 5
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    gradient_clip = 1.0
    num_workers = 0

print("Device:", CFG.device)
print("Model:", CFG.model_name)

c:\Users\Juan David\Downloads\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Model: microsoft/deberta-v3-base


## 2. Carga de datos

In [2]:
train = pd.read_csv('../data/train.csv')
eval_df = pd.read_csv('../data/eval.csv')

print(f"Train shape: {train.shape}")
print(f"Eval shape:  {eval_df.shape}")

le = LabelEncoder()
train['label'] = le.fit_transform(train['decade'])
num_classes = len(le.classes_)
print(f"Num classes: {num_classes}")

class_names = le.classes_
label_map = dict(enumerate(class_names))

Train shape: (31403, 2)
Eval shape:  (3490, 2)
Num classes: 39


## 3. Exploración Temporal

Se analizan patrones lingüísticos a través de las décadas para entender qué señales temporales existen en los datos.

### 3.1 Distribución de décadas

Verificar el balanceo de clases.

In [3]:
print("Distribucion de decadas:")
print(train['decade'].value_counts().sort_index())
print(f"\nDecadas unicas: {train['decade'].nunique()}")
print(f"Balance relativo: {train['decade'].value_counts().min() / train['decade'].value_counts().max():.3f}")

Distribucion de decadas:
decade
150    786
151    812
152    785
153    775
154    830
155    836
156    792
157    827
158    778
159    802
160    848
161    787
162    808
163    827
164    804
165    814
166    779
167    831
168    822
169    771
170    833
171    816
172    842
173    802
174    807
175    817
176    754
177    782
178    831
179    809
180    825
181    795
182    808
183    794
184    802
185    803
186    773
187    787
188    809
Name: count, dtype: int64

Decadas unicas: 39
Balance relativo: 0.889


### 3.2 Longitud de textos por década

Las épocas más tempranas suelen tener párrafos más cortos o fragmentados por el escaneo OCR. Esto puede ser una señal discriminativa.

In [4]:
train['text_len'] = train['text'].apply(len)
train['word_count'] = train['text'].apply(lambda x: len(x.split()))
train['sentence_count'] = train['text'].apply(lambda x: len(re.split(r'[.!?]+', x)))

print("Estadisticas por decada:")
print(train.groupby('decade')[['text_len', 'word_count', 'sentence_count']].mean().round(1))

Estadisticas por decada:
        text_len  word_count  sentence_count
decade                                      
150        280.9        45.7             4.0
151        853.2       115.7            11.8
152        385.3        55.7             4.8
153        678.9        96.7             8.9
154        674.6       100.2             7.4
155        670.2        95.9             7.1
156        407.8        62.9             5.7
157        549.7        79.8             7.1
158        592.8        91.6             5.4
159        601.8        91.7             6.0
160        497.9        78.4             6.1
161        537.0        82.2             5.9
162        549.7        85.1             6.0
163        546.6        86.9             6.8
164        575.6        89.7             6.4
165        471.3        74.2             6.0
166        474.1        74.2             5.7
167        526.3        83.2             6.2
168        540.6        87.6             5.5
169        548.9        88.5  

### 3.3 Palabras más discriminativas por década

Se usa TF-IDF para identificar qué términos son característicos de cada época. Esto revela vocabulario histórico, ortografía antigua y cambios léxicos.

In [5]:
top_k = 10
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(train['text'])

feature_names = tfidf.get_feature_names_out()

for decade in [150, 165, 180]:
    mask = train['decade'] == decade
    if mask.sum() == 0:
        continue
    centroid = tfidf_matrix[mask.values].mean(axis=0).A1
    top_idx = centroid.argsort()[-top_k:][::-1]
    top_words = [feature_names[i] for i in top_idx]
    print(f"Decada {decade}: {', '.join(top_words)}")

Decada 150: de, la, que, del, carta, en, su, de la, el, carta de
Decada 165: de, que, el, la, en, los, fe, no, fu, con
Decada 180: de, que, la, en, el, se, los, su, las, por


### 3.4 Distribución de puntuación

La puntuación histórica evoluciona con el tiempo. Signos como el punto y coma (;), la raya (—) o los dos puntos (:) aparecen con frecuencias distintas según la época.

In [6]:
punctuation_marks = [';', ':', '—', '(', ')', '"', '«', '»', '!']

for p in punctuation_marks:
    train[f'count_{p}'] = train['text'].apply(lambda x: x.count(p))

print("Puntuacion promedio por decada:")
print(train.groupby('decade')[[f'count_{p}' for p in punctuation_marks]].mean().round(2))

Puntuacion promedio por decada:
        count_;  count_:  count_—  count_(  count_)  count_"  count_«  \
decade                                                                  
150        0.20     0.57     0.10     0.27     0.21     0.02     0.10   
151        0.96     4.44     0.01     1.07     0.40     0.11     0.32   
152        0.34     1.22     0.13     0.19     0.13     0.17     0.22   
153        0.45     4.27     0.15     0.56     0.36     0.07     0.18   
154        0.38     4.08     0.04     0.45     0.21     0.07     0.26   
155        0.48     2.17     0.03     0.29     0.27     0.08     0.24   
156        0.28     1.17     0.03     0.26     0.14     0.09     0.16   
157        0.40     1.38     0.07     0.46     0.25     0.11     0.19   
158        0.42     1.33     0.02     0.40     0.26     0.08     0.19   
159        0.43     1.22     0.01     0.36     0.24     0.10     0.16   
160        0.36     0.93     0.03     0.32     0.21     0.12     0.22   
161        0.45    

### 3.5 Rareza léxica (Type-Token Ratio)

Mide la diversidad del vocabulario. Décadas con mayor riqueza léxica pueden reflejar géneros textuales diferentes.

In [7]:
train['ttr'] = train['text'].apply(
    lambda x: len(set(x.lower().split())) / max(len(x.split()), 1)
)

print("Type-Token Ratio promedio por decada:")
print(train.groupby('decade')['ttr'].mean().round(3))

Type-Token Ratio promedio por decada:
decade
150    0.827
151    0.899
152    0.864
153    0.906
154    0.871
155    0.871
156    0.874
157    0.878
158    0.849
159    0.850
160    0.862
161    0.862
162    0.843
163    0.856
164    0.854
165    0.868
166    0.865
167    0.851
168    0.837
169    0.839
170    0.857
171    0.830
172    0.816
173    0.832
174    0.812
175    0.802
176    0.797
177    0.806
178    0.798
179    0.802
180    0.816
181    0.811
182    0.809
183    0.819
184    0.797
185    0.809
186    0.821
187    0.818
188    0.807
Name: ttr, dtype: float64


## 4. Preprocesamiento Minimalista

A diferencia de tareas NLP tradicionales, en este problema las señales estilísticas, ortográficas y de puntuación contienen información temporal valiosa.

Por esta razón:
- **no se eliminan stopwords**
- **no se aplica stemming ni lematización**
- **no se convierte todo a minúsculas** (se normaliza solo lo necesario)

Solo se corrigen artefactos extremos de OCR y espacios múltiples.

In [8]:
def minimal_clean(text):
    # Eliminar saltos de linea multiples
    text = re.sub(r'\n{3,}', '\n', text)
    # Espacios multiples
    text = re.sub(r'\s+', ' ', text)
    # Caracteres corruptos extremos (no imprimibles)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text.strip()

train['text_clean'] = train['text'].apply(minimal_clean)
eval_df['text_clean'] = eval_df['text'].apply(minimal_clean)

print("Muestra de texto limpio (decada 150):")
print(train[train['decade'] == 150]['text_clean'].iloc[0][:500])

Muestra de texto limpio (decada 150):
efiotnl fiiT’e^ pt\»tf)e 4 trCe et lleene.^^ta^ 41 tT»íi A*e(leee A(ittc(«t>iieii|l>le iié <oii|ette *iW íléiW^* temer pw tnner>>ellmpí«íK>


## 5. Features Temporales Manuales

Se extraen señales lingüísticas explícitas que el transformer podría no capturar completamente. Estas features se concatenan con el embedding `[CLS]` antes de la clasificación.

In [9]:
def extract_temporal_features(text):
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    words = text.split()
    char_count = len(text)
    word_count = len(words)
    sentence_count = len(sentences) if sentences else 1

    feats = []

    # Longitud promedio de oracion
    feats.append(np.mean([len(s.split()) for s in sentences]) if sentences else 0)

    # Longitud promedio de palabra
    feats.append(np.mean([len(w) for w in words]) if words else 0)

    # Ratio de mayusculas
    feats.append(sum(1 for c in text if c.isupper()) / max(char_count, 1))

    # Densidad de puntuacion
    total_punct = sum(1 for c in text if c in string.punctuation)
    feats.append(total_punct / max(char_count, 1))

    # Conteos de puntuacion especifica
    for mark in [';', ':', '—', '(', ')', '«', '»']:
        feats.append(text.count(mark) / max(word_count, 1))

    # Type-Token Ratio
    unique_words = len(set(w.lower() for w in words))
    feats.append(unique_words / max(word_count, 1))

    # Proporcion de palabras largas (>8 caracteres)
    long_words = sum(1 for w in words if len(w) > 8)
    feats.append(long_words / max(word_count, 1))

    return np.array(feats, dtype=np.float32)

# Calcular features para todos los textos
train_features = np.array([extract_temporal_features(t) for t in train['text_clean']])
eval_features = np.array([extract_temporal_features(t) for t in eval_df['text_clean']])

print(f"Feature vector dimension: {train_features.shape[1]}")
print(f"Train features shape: {train_features.shape}")
print(f"Eval features shape:  {eval_features.shape}")

Feature vector dimension: 13
Train features shape: (31403, 13)
Eval features shape:  (3490, 13)


## 6. Tokenización y Dataset

Se usa DeBERTa-v3-base con `max_length=256`. La clase `TemporalDataset` retorna `input_ids`, `attention_mask`, `features` (manuales) y opcionalmente `labels`.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

class TemporalDataset(Dataset):
    def __init__(self, texts, features, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'features': self.features[idx],
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

# Verificar
dummy = TemporalDataset(
    train['text_clean'].values[:2],
    train_features[:2],
    labels=train['label'].values[:2],
    tokenizer=tokenizer,
    max_len=CFG.max_len
)
batch = dummy[0]
print("input_ids:", batch['input_ids'].shape)
print("attention_mask:", batch['attention_mask'].shape)
print("features:", batch['features'].shape)
print("labels:", batch['labels'].shape)

input_ids: torch.Size([256])
attention_mask: torch.Size([256])
features: torch.Size([13])
labels: torch.Size([])


## 7. Modelo: DeBERTa + Features Temporales

La arquitectura combina:
1. **DeBERTa-v3-base** como encoder Transformer
2. **Features temporales manuales** (11 dimensiones) extraídas del texto
3. **Clasificador MLP** que recibe la concatenación de ambos

De esta forma el modelo aprende tanto representaciones semánticas profundas como señales estilísticas explícitas.

In [11]:
class TemporalClassifier(nn.Module):
    def __init__(self, model_name, num_classes, num_features, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + num_features, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, features):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        combined = torch.cat([cls_output, features], dim=1)
        logits = self.classifier(combined)
        return logits

# Verificacion
dummy_model = TemporalClassifier(CFG.model_name, num_classes, train_features.shape[1], dropout=0.3)
dummy_model.eval()
with torch.no_grad():
    out = dummy_model(
        batch['input_ids'].unsqueeze(0),
        batch['attention_mask'].unsqueeze(0),
        batch['features'].unsqueeze(0)
    )
print("Output shape:", out.shape)
print("Total params:", sum(p.numel() for p in dummy_model.parameters()))
print("Trainable params:", sum(p.numel() for p in dummy_model.parameters() if p.requires_grad))

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 7080.80it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok

Output shape: torch.Size([1, 39])
Total params: 184373287
Trainable params: 184373287


## 8. Entrenamiento

### 8.1 Funciones auxiliares

Se define el loop de entrenamiento con label smoothing y gradient clipping.

In [12]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, features)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask, features)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

### 8.2 Entrenamiento individual (prueba)

Entrenamiento en un solo split para verificar que todo funciona antes de lanzar el K-Fold completo.

In [ ]:
def train_single_fold(train_idx, val_idx, fold):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}")
    print(f"{'='*50}")

    X_train_fold = train['text_clean'].values[train_idx]
    y_train_fold = train['label'].values[train_idx]
    X_val_fold   = train['text_clean'].values[val_idx]
    y_val_fold   = train['label'].values[val_idx]
    feat_train_fold = train_features[train_idx]
    feat_val_fold   = train_features[val_idx]

    train_ds = TemporalDataset(X_train_fold, feat_train_fold, y_train_fold, tokenizer, CFG.max_len)
    val_ds   = TemporalDataset(X_val_fold,   feat_val_fold,   y_val_fold,   tokenizer, CFG.max_len)
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

    model = TemporalClassifier(CFG.model_name, num_classes, train_features.shape[1], dropout=0.3).to(CFG.device)

    optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    total_steps = len(train_loader) * CFG.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * CFG.warmup_ratio),
        num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

    best_acc = 0.0
    best_model_state = None
    history = []

    for epoch in range(CFG.epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, CFG.device)
        val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, CFG.device)

        history.append({
            'epoch': epoch+1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'lr': optimizer.param_groups[0]['lr']
        })

        print(f"  Epoch {epoch+1}/{CFG.epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_state = deepcopy(model.state_dict())

    model.load_state_dict(best_model_state)
    _, _, oof_preds, oof_labels = eval_epoch(model, val_loader, criterion, CFG.device)

    return model, oof_preds, oof_labels, best_acc, history

skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
folds = list(skf.split(train['text_clean'].values, train['label']))

# Probar primer fold
model_f0, oof_preds_0, oof_labels_0, acc_0, hist_0 = train_single_fold(*folds[0], 0)
print(f"\nFold 0 validation accuracy: {acc_0:.4f}")


Fold 1


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 6741.12it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok

## 9. Validación Cruzada Completa (5 Folds)

Se entrena en los 5 folds estratificados. Se almacenan:
- Accuracy por fold
- Matriz de confusión por fold
- OOF predictions para el ensemble
- Mejor checkpoint de cada fold

In [ ]:
all_oof_preds = np.zeros((len(train), num_classes))
all_models = []
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(folds):
    model, oof_preds, oof_labels, best_acc, history = train_single_fold(train_idx, val_idx, fold)
    all_models.append(model)

    # OOF probabilities (softmax sobre logits)
    model.eval()
    val_ds = TemporalDataset(
        train['text_clean'].values[val_idx],
        train_features[val_idx],
        tokenizer=tokenizer, max_len=CFG.max_len
    )
    val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False)
    fold_logits = []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch['input_ids'].to(CFG.device),
                batch['attention_mask'].to(CFG.device),
                batch['features'].to(CFG.device)
            )
            fold_logits.append(logits.cpu().numpy())
    fold_logits = np.concatenate(fold_logits)
    fold_probs = np.exp(fold_logits) / np.exp(fold_logits).sum(axis=1, keepdims=True)
    all_oof_preds[val_idx] = fold_probs

    fold_scores.append(best_acc)
    print(f"\n>>> Fold {fold+1} completed. Best val acc: {best_acc:.4f}")
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*50}")
print(f"Cross-Validation Results ({CFG.n_folds} folds):")
for i, acc in enumerate(fold_scores):
    print(f"  Fold {i+1}: {acc:.4f}")
print(f"  Mean CV Accuracy: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")

## 10. Ensemble de Folds

Se combinan las predicciones de los 5 folds promediando los logits. Esto reduce la varianza y suele mejorar el accuracy en Kaggle.

In [ ]:
# OOF ensemble accuracy
oof_preds_final = all_oof_preds.argmax(axis=1)
oof_accuracy = accuracy_score(train['label'].values, oof_preds_final)
print(f"OOF Ensemble Accuracy: {oof_accuracy:.4f}")

# Mejora respecto al promedio individual
mean_fold = np.mean(fold_scores)
print(f"Mean individual fold: {mean_fold:.4f}")
print(f"Ensemble improvement: {oof_accuracy - mean_fold:+.4f}")

## 11. Análisis de Errores

Se examinan las confusiones del modelo para entender qué décadas son difíciles de distinguir. Esto guía las mejoras futuras.

In [ ]:
cm = confusion_matrix(train['label'].values, oof_preds_final)
print("Confusion Matrix (primeras 10 clases):")
print(pd.DataFrame(cm[:10, :10], index=class_names[:10], columns=class_names[:10]))

# Errores mas frecuentes
n_classes = len(class_names)
error_pairs = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            error_pairs.append((cm[i, j], class_names[i], class_names[j]))

error_pairs.sort(reverse=True)
print("\nPares de decadas mas confundidas:")
for count, true, pred in error_pairs[:10]:
    print(f"  {true} -> {pred}: {count} muestras")

### 11.1 Ejemplos de textos mal clasificados

Examinar casos concretos donde el modelo falla ayuda a identificar sesgos o patrones no capturados.

In [ ]:
misclassified_mask = oof_preds_final != train['label'].values
misclassified_idx = np.where(misclassified_mask)[0]

print(f"Total misclassified: {len(misclassified_idx)} / {len(train)} ({len(misclassified_idx)/len(train)*100:.1f}%)")

# Mostrar algunos ejemplos
for idx in misclassified_idx[:5]:
    true_decade = class_names[train['label'].values[idx]]
    pred_decade = class_names[oof_preds_final[idx]]
    text_preview = train['text_clean'].values[idx][:200]
    print(f"\nTrue: {true_decade} | Pred: {pred_decade}")
    print(f"Text: {text_preview}...")

## 12. Generación de Submission

Se promedian las predicciones de todos los modelos entrenados en cada fold para producir la predicción final sobre el conjunto de evaluación.

In [ ]:
print(f"Generando predicciones con ensemble de {len(all_models)} modelos...")

eval_dataset = TemporalDataset(
    eval_df['text_clean'].values,
    eval_features,
    labels=None,
    tokenizer=tokenizer,
    max_len=CFG.max_len
)
eval_loader = DataLoader(eval_dataset, batch_size=CFG.batch_size, shuffle=False)

# Promedio de logits de todos los folds
all_fold_logits = []

for fold, model in enumerate(all_models):
    model.eval()
    fold_logits = []
    with torch.no_grad():
        for batch in eval_loader:
            logits = model(
                batch['input_ids'].to(CFG.device),
                batch['attention_mask'].to(CFG.device),
                batch['features'].to(CFG.device)
            )
            fold_logits.append(logits.cpu().numpy())
    fold_logits = np.concatenate(fold_logits)
    all_fold_logits.append(fold_logits)
    print(f"  Fold {fold+1} logits: {fold_logits.shape}")

# Average ensemble
final_logits = np.mean(all_fold_logits, axis=0)
final_preds = final_logits.argmax(axis=1)
predicted_decades = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    'id': eval_df['id'],
    'answer': predicted_decades
})

submission.to_csv('submission.csv', index=False)
print(f"\nSubmission guardada: {submission.shape}")
print(submission.head(10))

## 13. Mejoras Avanzadas (Opcionales)

Las siguientes secciones contienen código comentado para experimentar con técnicas más avanzadas.

### 13.1 Fine-tuning Parcial

Si el entrenamiento completo es muy lento, se pueden congelar las primeras capas de DeBERTa y solo entrenar las últimas.

In [ ]:
# # Congelar backbone y descongelar solo ultimas 2 capas
# for param in model.backbone.parameters():
#     param.requires_grad = False
# for layer in model.backbone.encoder.layer[-2:]:
#     for param in layer.parameters():
#         param.requires_grad = True
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in model.parameters())
# print(f"Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)")

### 13.2 Rama Char-Level (CNN)

Una rama adicional que procesa el texto a nivel de caracteres puede capturar patrones ortográficos históricos que los subword tokens de BPE pierden.

In [ ]:
# class CharCNN(nn.Module):
#     def __init__(self, vocab_size=128, embed_dim=32, num_filters=128, kernel_sizes=[3, 5, 7]):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim)
#         self.convs = nn.ModuleList([
#             nn.Conv1d(embed_dim, num_filters, k, padding=k//2)
#             for k in kernel_sizes
#         ])
#         self.dropout = nn.Dropout(0.3)
#
#     def forward(self, x):  # x: (batch, max_char_len)
#         x = self.embedding(x).permute(0, 2, 1)
#         x = [torch.max(torch.relu(conv(x)), dim=2)[0] for conv in self.convs]
#         x = torch.cat(x, dim=1)
#         return self.dropout(x)

### 13.3 Historical MLM Pretraining

Pre-entrenar DeBERTa con MLM en corpus históricos (Project Gutenberg, archivos coloniales) antes del fine-tuning puede mejorar la representación del lenguaje antiguo. Este paso requiere un dataset externo.

In [ ]:
# from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
#
# # Cargar corpus historico
# # historical_texts = [...]  # textos de Project Gutenberg, archivos, etc.
#
# mlm_dataset = TemporalDataset(historical_texts, features=None, tokenizer=tokenizer)
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
#
# training_args = TrainingArguments(
#     output_dir='./historical-mlm',
#     per_device_train_batch_size=8,
#     num_train_epochs=3,
#     learning_rate=5e-5,
#     save_total_limit=1,
# )
#
# trainer = Trainer(
#     model=AutoModelForMaskedLM.from_pretrained(CFG.model_name),
#     args=training_args,
#     data_collator=data_collator,
#     train_dataset=mlm_dataset,
# )
# trainer.train()
# # Guardar y reemplazar backbone con el modelo pre-entrenado

### 13.4 Tabla de Experimentos

Mantener un registro de los experimentos realizados para mantener trazabilidad.

In [ ]:
# TABLA DE EXPERIMENTOS (completar manualmente)
# | Modelo | Preproc | Features | Folds | CV Acc | Public | Notas |
# |--------|---------|----------|-------|--------|--------|-------|
# | deberta-v3-base | minimal | si (11) | 5 | - | - | baseline competitivo |